# 🚀 jDroid-X-Desk: Centralized Cloud Relay & Remote Access Server v3.0
**Google Colab Independent Relay & Remote Desktop Bridge | Subsidiary of jDroid-X Studio**

This notebook runs as a centralized signaling, frame buffer, and input relay server in Google Colab.
- **Zero Configuration**: Bridges **Desk 1**, **Desk 2**, and **Desk 3** across NAT, CGNAT, firewalls, and mobile hotspots.
- **Wake-on-Ping Active**: Listens on `/ping`, `/api/relay/ping`, and `/api/relay/wake` to auto-wake when contacted by `jDroid-X-Desk.exe`.
- **Autonomous Watchdog**: Background watchdog daemon auto-restarts the relay and keeps the Colab runtime active.
- **Zero Cost**: Runs entirely on Google Colab's high-bandwidth cloud network.

### Step 1: Install Tunnel & Dependencies

In [ ]:
# Download Cloudflare Tunnel binary for secure public ingress
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print('✅ Cloudflared tunnel binary is ready!')

### Step 2: Fetch or Initialize Relay Server & Watchdog

In [ ]:
# Clone or fetch latest jDroid-X-Colab-Relay-Server.py and colab-watchdog-keepalive.py
!wget -q -O jDroid-X-Colab-Relay-Server.py https://raw.githubusercontent.com/jDroid-X/jDroid-X-Desk/main/jDroid-X-Colab%20Server/jDroid-X-Colab-Relay-Server.py || \
 wget -q -O jDroid-X-Colab-Relay-Server.py https://raw.githubusercontent.com/jDroid-X/jDroid-X-Desk/main/colab_relay_server.py

!wget -q -O colab-watchdog-keepalive.py https://raw.githubusercontent.com/jDroid-X/jDroid-X-Desk/main/jDroid-X-Colab%20Server/colab-watchdog-keepalive.py || true

print('✅ Relay Server & Watchdog scripts loaded!')

### Step 3: Launch Central Relay Server & Autonomous Watchdog
Click Play on the cell below to start the server on port 8088.

In [ ]:
import subprocess, time

# 1. Start Relay Server
server_proc = subprocess.Popen(['python3', 'jDroid-X-Colab-Relay-Server.py', '8088'])
time.sleep(2)

# 2. Start Watchdog Keep-Alive Daemon
try:
    watchdog_proc = subprocess.Popen(['python3', 'colab-watchdog-keepalive.py'])
except Exception as e:
    print('Watchdog daemon note:', e)

print('\n' + '='*60)
print('  🚀 jDroid-X Colab Cloud Relay is ONLINE on Port 8088!')
print('  Wake-on-Ping: Active on /ping and /api/relay/ping')
print('  Starting Cloudflare Tunnel to expose Public URL...')
print('='*60 + '\n')

# 3. Expose Tunnel (Displays public https://*.trycloudflare.com URL)
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8088

### (Optional) Step 4: Run Chrome Remote Desktop Headless Host in Colab
If you want direct interactive desktop control of this Colab virtual machine via Google Remote Desktop:

In [ ]:
#@title Setup Chrome Remote Desktop in Colab
CRD_AUTH_CODE = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}

if CRD_AUTH_CODE:
    !apt update -qq
    !wget -q https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
    !dpkg --install chrome-remote-desktop_current_amd64.deb || apt install -y --fix-broken
    !DEBIAN_FRONTEND=noninteractive apt install -y xfce4 desktop-base xfce4-terminal
    !bash -c 'echo "exec /etc/X11/Xsession /usr/bin/xfce4-session" > /etc/chrome-remote-desktop-session'
    !useradd -m -s /bin/bash user
    !echo 'user:root' | chpasswd
    !adduser user sudo
    !adduser user chrome-remote-desktop
    cmd = f'{CRD_AUTH_CODE} --pin={PIN}'
    !su - user -c '{cmd}'
    !service chrome-remote-desktop start
    print('✅ Chrome Remote Desktop Started! Check remotedesktop.google.com/access')
else:
    print('To run CRD, paste the auth command from remotedesktop.google.com/headless into CRD_AUTH_CODE above.')